### Requirements 

#### .env file

```bash
MILVUS_URL = # your milvus server url
MILVUS_TOKEN = # your milvus access token
MILVUS_COLLECTION_NAME = # milvus collection name
```

In [1]:
import os
import sys 
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)
ROOT = Path.cwd().parent.parent.parent.resolve()

print(f"ROOT: {ROOT}")
sys.path.append(str(ROOT))

ROOT: /users/devin/Dev/financial-document-based-agent-system


### Milvus server Info

In [2]:
import pymilvus
import os
from urllib.parse import urlparse

In [11]:
milvus_url = os.environ.get("MILVUS_URL")
milvus_token = os.environ.get("MILVUS_TOKEN")
milvus_collection_name = os.environ.get("MILVUS_COLLECTION_NAME")

if not milvus_url:
    raise RuntimeError("MILVUS_URL environment variable not set")

# parse host/port if needed
u = urlparse(milvus_url if "://" in milvus_url else f"//{milvus_url}")
host = u.hostname or milvus_url
port = u.port or 19530

# try connecting (try uri first, then host/port with optional token as password)
errors = []
try:
    pymilvus.connections.connect(uri=milvus_url)
except Exception as e:
    errors.append(e)
    try:
        if milvus_token:
            pymilvus.connections.connect(host=host, port=str(port), password=milvus_token)
        else:
            pymilvus.connections.connect(host=host, port=str(port))
    except Exception as e2:
        errors.append(e2)
        raise RuntimeError("Failed to connect to Milvus", errors)

print("Connected to:", milvus_url)
print("Collections:", pymilvus.utility.list_collections())

Connected to: http://localhost:19530
Collections: ['xml_documentsv5', 'xml_documents_bge', 'e2e_1st', 'agent_collection_9d83744d893e43f1a5f98a67b204efc8', 'agent_collection_9930bc63dd134a03a5e3bcdaddc45101', 'e2e_2st', 'agent_collection_a5a623ff0db54a2d86640c7d3c66b6b2', 'xml_documents_nomic', 'agent_collection_8d5bd5d920ea48e5827b45705864727c', 'agent_collection_825fdc93e1bd46ef979abbc73da4bee2', 'agent_collection_da03213747de491cb674e87d550bc1b8', 'agent_collection_03b4f13a579941c1a880f236ab1a8b2a', 'agent_collection_b52afc800301461497895b9ece5cc926', 'agent_collection_0994246b1f7f4a38b1c5f25cf6313c99', 'agent_collection_e8986c11ca894481bc055bf0d1a769da', 'agent_collection_07c82d8de50d40a99d0bab41bd497918', 'agent_collection_618e846b87bc40a7a50d752765a563b7', 'agent_collection_3eb2d5d9309b4a37b581cabbf24a96d1', 'agent_collection_c98c9ef0b6744f79be365169eca3791d', 'agent_collection_e6aab4df2bbe47868afd76ae750703a5', 'agent_collection_02a3523a75764604b9cef115ce973b75', 'test_collection

In [12]:
milvus_collection_name in pymilvus.utility.list_collections()

False

### Class info

In [13]:
from cgcore.vectordb.milvus import MilvusDB
from cgcore.configs.vectordb.milvus import MilvusConfig

In [14]:
config = MilvusConfig(**{
    "collection_name": milvus_collection_name,
    "dimensions": 1536,
})

milvus_db = MilvusDB(config)

MilvusClient connected.
pymilvus ORM connected to localhost:19530 for setup.
Collection 'financial_documents' created.


In [15]:
config.collection_name

'financial_documents'

#### Insert text

In [16]:
question = "What is Milvus?"
embedding = [0.0] * 1536  # Dummy embedding for testing
content = "Milvus is an open-source vector database."
docid = "test_paper_1"

milvus_db.insert([
    {
        "content": content, 
        "vector": embedding, 
        "doc_id": docid, 
        "meta_data": {"source": "test_source"}
    },
    {
        "content": "Milvus supports efficient similarity search.", 
        "vector": embedding, 
        "doc_id": "test_paper_2", 
        "meta_data": {"source": "test_source"}
    }
])

True

#### Retrived text

In [17]:
milvus_db.vector_search(embedding, top_k=2)

[DEBUG FULL CONTENT] ID: 463710524979398996
[DEBUG FULL CONTENT] Content: 'Milvus is an open-source vector database.'
[DEBUG FULL CONTENT] Content length: 41
[DEBUG FULL CONTENT] Metadata: {'source': 'test_source'}
[DEBUG] Formatted chunk: content=Milvus is an open-source vector database...., distance=0.0
[DEBUG FULL CONTENT] ID: 463710524979398997
[DEBUG FULL CONTENT] Content: 'Milvus supports efficient similarity search.'
[DEBUG FULL CONTENT] Content length: 44
[DEBUG FULL CONTENT] Metadata: {'source': 'test_source'}
[DEBUG] Formatted chunk: content=Milvus supports efficient similarity search...., distance=0.0
[DEBUG] Returning 2 formatted results


[{'content': 'Milvus is an open-source vector database.',
  '_id': '463710524979398996',
  'meta_data': {'source': 'test_source'},
  'distance': 0.0},
 {'content': 'Milvus supports efficient similarity search.',
  '_id': '463710524979398997',
  'meta_data': {'source': 'test_source'},
  'distance': 0.0}]

### Drop collection if necessary